# Week 11 extra: Grad-CAM on the 3-class models
**TMI-102 | PaAC, IIT Roorkee** — runs on Kaggle (GPU on)

Your Week 7 Grad-CAM used the old 2-class model and had no merger. This notebook redoes it with the **3-class** models: one heatmap per class (this is panel C of the Week 11 summary figure), a Full FT vs LoRA comparison on mergers, and a look at wrong predictions.

**Setup:** add (1) the Galaxy Zoo competition data and (2) the Kaggle dataset that holds `w5_full_3class.keras` and `w6_lora_3class.keras` (the same one you used for Week 9). Turn the GPU on and run top to bottom.

Examples are picked **at random with a fixed seed**, not hand-picked, so the figure is not cherry-picked.


In [ ]:
# ── Build Galaxy_dataset_3class from the competition files (runs once; skipped if the folder already exists) ──
import os, zipfile
import numpy as np, pandas as pd

COMP = "/kaggle/input/competitions/galaxy-zoo-the-galaxy-challenge"
BUILD_DIR = "/kaggle/working/Galaxy_dataset_3class"
ELL_T, SPI_T, MERGER_T, SEED = 0.80, 0.80, 0.30, 42     # merger = Class8.6 >= 0.30 (change here if mentors say otherwise)

if not os.path.isdir(BUILD_DIR + "/train"):
    from sklearn.model_selection import train_test_split
    df = pd.read_csv(f"{COMP}/training_solutions_rev1.zip", compression="zip")

    # --- elliptical + spiral: EXACTLY the Week 3 recipe (same thresholds, same seed, same split) ---
    df["class"] = np.select([df["Class1.1"] >= ELL_T, df["Class1.2"] >= SPI_T], ["elliptical", "spiral"], default="Tie")
    filtered = df[df["class"].isin(["elliptical", "spiral"])].copy()
    min_cnt = min(filtered["class"].value_counts())
    balanced = pd.concat([
        filtered[filtered["class"] == "spiral"].sample(n=min_cnt, random_state=SEED),
        filtered[filtered["class"] == "elliptical"].sample(n=min_cnt, random_state=SEED)
    ]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    tr, tmp = train_test_split(balanced, test_size=0.30, random_state=SEED, stratify=balanced["class"])
    va, te = train_test_split(tmp, test_size=0.50, random_state=SEED, stratify=tmp["class"])

    # --- merger: simply ADDED as a third class. Galaxies already used as elliptical/spiral are skipped,
    #     so one image never has two labels and the two old classes stay exactly as they were. ---
    mer_all = df[df["Class8.6"] >= MERGER_T]
    mer = mer_all[~mer_all["GalaxyID"].isin(balanced["GalaxyID"])].copy()
    mer["class"] = "merger"
    print(f"merger candidates {len(mer_all)} -> {len(mer)} after skipping {len(mer_all) - len(mer)} already used as elliptical/spiral")
    mtr, mtmp = train_test_split(mer, test_size=0.30, random_state=SEED)
    mva, mte = train_test_split(mtmp, test_size=0.50, random_state=SEED)

    splits = [(tr, "train"), (va, "val"), (te, "test"), (mtr, "train"), (mva, "val"), (mte, "test")]
    for s in ["train", "val", "test"]:
        for c in ["elliptical", "merger", "spiral"]:
            os.makedirs(f"{BUILD_DIR}/{s}/{c}", exist_ok=True)
    target, rows = {}, []
    for d, s in splits:
        for gid, c in zip(d["GalaxyID"], d["class"]):
            target[f"{gid}.jpg"] = (s, c); rows.append((gid, c, s))

    n_ext = 0
    with zipfile.ZipFile(f"{COMP}/images_training_rev1.zip") as z:
        for member in z.namelist():
            base = os.path.basename(member)
            if base in target:
                s, c = target[base]
                with open(f"{BUILD_DIR}/{s}/{c}/{base}", "wb") as f:
                    f.write(z.read(member))
                n_ext += 1
    man = pd.DataFrame(rows, columns=["GalaxyID", "cls", "split"])
    man.to_csv(f"{BUILD_DIR}/manifest.csv", index=False)
    print("extracted", n_ext, "of", len(man))
    print(man.groupby(["split", "cls"]).size().unstack())
    print("Check vs Week 3: elliptical/spiral should be 5692 / 1220 / 1220 each.")
else:
    print("Dataset already built at", BUILD_DIR)

In [ ]:
import glob, os, numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

@tf.keras.utils.register_keras_serializable(package='tmi102')
class LoRADense(layers.Layer):
    """Dense layer with a LoRA adapter: output = x@W + b + (x@A@B) * (alpha/rank).  W, b frozen; A, B trainable."""
    def __init__(self, units, rank=4, alpha=32, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units, self.rank, self.alpha = units, rank, alpha
        self.scaling = alpha / rank
        self.activation_fn = tf.keras.activations.get(activation)

    def build(self, input_shape):
        input_dim = input_shape[-1]
        self.W = self.add_weight(name='kernel', shape=(input_dim, self.units), initializer='glorot_uniform', trainable=False)
        self.b = self.add_weight(name='bias', shape=(self.units,), initializer='zeros', trainable=False)
        self.A = self.add_weight(name='lora_A', shape=(input_dim, self.rank), initializer='glorot_normal', trainable=True)
        self.B = self.add_weight(name='lora_B', shape=(self.rank, self.units), initializer='zeros', trainable=True)

    def call(self, x):
        h = tf.matmul(x, self.W) + self.b
        h = h + tf.matmul(tf.matmul(x, self.A), self.B) * self.scaling
        return self.activation_fn(h) if self.activation_fn else h

    def get_config(self):
        cfg = super().get_config()
        cfg.update(units=self.units, rank=self.rank, alpha=self.alpha,
                   activation=tf.keras.activations.serialize(self.activation_fn))
        return cfg

def find(name):
    return glob.glob(f'/kaggle/input/**/{name}', recursive=True)[0]

full = tf.keras.models.load_model(find('w5_full_3class.keras'), custom_objects={'LoRADense': LoRADense})
lora = tf.keras.models.load_model(find('w6_lora_3class.keras'), custom_objects={'LoRADense': LoRADense})

test_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
    f'{BUILD_DIR}/test', target_size=(224, 224), batch_size=32, class_mode='categorical', shuffle=False)
CLASS_NAMES = [c for c, _ in sorted(test_gen.class_indices.items(), key=lambda kv: kv[1])]
y_true = test_gen.classes
probs_full = full.predict(test_gen, verbose=0); test_gen.reset()
probs_lora = lora.predict(test_gen, verbose=0)
pred_full, pred_lora = probs_full.argmax(1), probs_lora.argmax(1)
print(CLASS_NAMES)
print(f"Full FT accuracy {np.mean(pred_full == y_true) * 100:.2f}% | LoRA accuracy {np.mean(pred_lora == y_true) * 100:.2f}%")


## Grad-CAM function

In [ ]:
def grad_cam(model, image, class_idx=None):
    """Grad-CAM for a model with one nested ResNet-50 and a small head. Returns (7x7 heatmap, class probabilities)."""
    resnet = [l for l in model.layers if isinstance(l, tf.keras.Model)][0]
    head = model.layers[model.layers.index(resnet) + 1:]
    with tf.GradientTape() as tape:
        features = resnet(image[np.newaxis], training=False)
        tape.watch(features)
        x = features
        for l in head:
            x = l(x)
        k = tf.argmax(x[0]) if class_idx is None else class_idx
        score = x[:, k]
    grads = tape.gradient(score, features)
    weights = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.maximum(tf.squeeze(features[0] @ weights[..., tf.newaxis]), 0)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy(), x.numpy()[0]

def load_galaxy(path):
    raw = np.array(tf.keras.utils.load_img(path, target_size=(224, 224))).astype('float32')
    return raw / 255.0, preprocess_input(raw.copy())      # (for display, for the model)

def overlay(image, heatmap, alpha=0.4):
    big = tf.image.resize(heatmap[..., np.newaxis], (224, 224)).numpy().squeeze()
    return np.clip(cm.jet(big)[:, :, :3] * alpha + image * (1 - alpha), 0, 1)

rng = np.random.default_rng(0)
print("Grad-CAM ready")


## 1. One heatmap per class (panel C)

In [ ]:
# One correctly classified random test galaxy per class, Full FT model -> panel C of the summary figure
fig, axes = plt.subplots(2, 3, figsize=(11, 7.5))
for j, cls in enumerate(CLASS_NAMES):
    correct = np.where((y_true == j) & (pred_full == j))[0]
    i = rng.choice(correct)
    img, mimg = load_galaxy(test_gen.filepaths[i])
    cam, p = grad_cam(full, mimg)
    axes[0, j].imshow(img); axes[0, j].set_title(f'{cls} (original)'); axes[0, j].axis('off')
    axes[1, j].imshow(overlay(img, cam)); axes[1, j].set_title(f'Grad-CAM, predicted {CLASS_NAMES[p.argmax()]} ({p.max() * 100:.0f}%)', fontsize=9); axes[1, j].axis('off')
plt.suptitle('Full fine-tuning: what the model looks at, one random correct example per class', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/gradcam_per_class.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()


## 2. Mergers: Full FT vs LoRA

In [ ]:
# 3 random merger galaxies: Full FT vs LoRA on the same images
merger_idx = rng.choice(np.where(y_true == CLASS_NAMES.index('merger'))[0], 3, replace=False)
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for r, i in enumerate(merger_idx):
    img, mimg = load_galaxy(test_gen.filepaths[i])
    cf, pf = grad_cam(full, mimg); cl, pl = grad_cam(lora, mimg)
    axes[r, 0].imshow(img); axes[r, 0].set_title('merger (original)'); axes[r, 0].axis('off')
    axes[r, 1].imshow(overlay(img, cf)); axes[r, 1].set_title(f'Full FT: {CLASS_NAMES[pf.argmax()]} ({pf.max() * 100:.0f}%)', fontsize=9); axes[r, 1].axis('off')
    axes[r, 2].imshow(overlay(img, cl)); axes[r, 2].set_title(f'LoRA: {CLASS_NAMES[pl.argmax()]} ({pl.max() * 100:.0f}%)', fontsize=9); axes[r, 2].axis('off')
plt.tight_layout(); plt.show()


## 3. Where the Full FT model goes wrong

In [ ]:
# Wrong predictions of the Full FT model: its two biggest confusions
def show_wrong(true_cls, pred_cls, n=2):
    t, p_ = CLASS_NAMES.index(true_cls), CLASS_NAMES.index(pred_cls)
    idx = np.where((y_true == t) & (pred_full == p_))[0]
    print(f"{true_cls} -> {pred_cls}: {len(idx)} images")
    fig, axes = plt.subplots(1, n * 2, figsize=(4 * n, 3.4))
    for k, i in enumerate(rng.choice(idx, n, replace=False)):
        img, mimg = load_galaxy(test_gen.filepaths[i])
        cam, p = grad_cam(full, mimg)
        axes[2 * k].imshow(img); axes[2 * k].set_title(f'true {true_cls}'); axes[2 * k].axis('off')
        axes[2 * k + 1].imshow(overlay(img, cam)); axes[2 * k + 1].set_title(f'pred {pred_cls} ({p.max() * 100:.0f}%)', fontsize=9); axes[2 * k + 1].axis('off')
    plt.tight_layout(); plt.show()

show_wrong('spiral', 'merger')
show_wrong('merger', 'spiral')
